# 1. Data Understanding

Ziel dieses Notebooks ist es, aus den Rohdaten im Ordner `data/raw/` einen monatlichen Random-Forest-Datensatz zu bauen, mit dem sich vorhersagen lässt, ob der GPR im Folgemonat steigt.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import LabelEncoder

# Projektpfade und Konstanten
BASE_DIR = Path(r"d:/Anwendungsprojekt")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
INDEX_PATH = RAW_DIR / "indexData.csv"
GPR_XLS_PATH = RAW_DIR / "data_gpr_daily_recent.xls"
GPR_XLSX_PATH = RAW_DIR / "data_gpr_daily_recent.xlsx"
DAILY_OUTPUT_PATH = PROCESSED_DIR / "stocks_gpr_daily_2001_2021.csv"
FINAL_OUTPUT_PATH = PROCESSED_DIR / "dataset_2001_2021.csv"

VALID_INDICES = [
    "NYA",
    "IXIC",
    "GSPTSE",
    "GDAXI",
    "N100",
    "SSMI",
    "N225",
    "HSI",
    "000001.SS",
    "399001.SZ",
    "KS11",
    "TWII",
]
EXCLUDED_INDICES = ["NSEI", "J203.JO"]
REGION_MAP = {
    "NYA": "USA",
    "IXIC": "USA",
    "GSPTSE": "Nordamerika",
    "GDAXI": "Europa",
    "N100": "Europa",
    "SSMI": "Europa",
    "N225": "Japan",
    "HSI": "China_HK",
    "000001.SS": "China_HK",
    "399001.SZ": "China_HK",
    "KS11": "Asien_Pazifik",
    "TWII": "Asien_Pazifik",
}
CRISIS_PERIODS = [
    ("2001-08", "2002-06"),
    ("2008-08", "2009-06"),
    ("2011-01", "2011-09"),
    ("2014-02", "2014-09"),
    ("2020-01", "2020-06"),
]
TRAIN_CUTOFF = pd.Period("2015-12", freq="M")
FINAL_START = pd.Period("2001-01", freq="M")
FINAL_END = pd.Period("2021-05", freq="M")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


def clean_column_names(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = (
        frame.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
    )
    return frame


def load_gpr_file() -> pd.DataFrame:
    if GPR_XLS_PATH.exists():
        gpr_path = GPR_XLS_PATH
        engine = "xlrd"
    elif GPR_XLSX_PATH.exists():
        gpr_path = GPR_XLSX_PATH
        engine = "openpyxl"
    else:
        raise FileNotFoundError(
            f"Keine GPR-Excel-Datei gefunden. Erwartet: {GPR_XLS_PATH.name} oder {GPR_XLSX_PATH.name} in {RAW_DIR}."
        )

    try:
        gpr_frame = pd.read_excel(gpr_path, engine=engine)
    except ImportError as exc:
        raise ImportError(
            f"Die Excel-Datei {gpr_path.name} konnte nicht gelesen werden. Bitte stelle sicher, dass die Engine '{engine}' installiert ist."
        ) from exc
    except Exception as exc:
        raise RuntimeError(f"Fehler beim Lesen von {gpr_path.name}: {exc}") from exc

    return gpr_frame


# 2. Data Preparation

In diesem Abschnitt werden die Rohdaten geladen, bereinigt, zeitlich normalisiert und zunächst als täglicher Join zusammengeführt.

In [2]:
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Die Datei {INDEX_PATH} wurde nicht gefunden.")

stocks = pd.read_csv(INDEX_PATH)
stocks = clean_column_names(stocks)

if "Date" not in stocks.columns or "Index" not in stocks.columns:
    raise ValueError("indexData.csv muss mindestens die Spalten 'Date' und 'Index' enthalten.")

stocks["Date"] = pd.to_datetime(stocks["Date"], errors="coerce")
stocks["Index"] = stocks["Index"].astype(str).str.strip()
stocks = stocks[~stocks["Index"].isin(EXCLUDED_INDICES)].copy()
stocks = stocks[stocks["Index"].isin(VALID_INDICES)].copy()
stocks = stocks.dropna(subset=["Date", "Close"])
stocks = stocks.sort_values(["Index", "Date"]).reset_index(drop=True)

gpr = load_gpr_file()
gpr = clean_column_names(gpr)

if "date" in gpr.columns and "Date" not in gpr.columns:
    gpr = gpr.rename(columns={"date": "Date"})

if "Date" not in gpr.columns:
    raise ValueError("Die GPR-Datei muss eine Spalte 'date' oder 'Date' enthalten.")

for required_column in ["GPRD", "GPRD_ACT", "GPRD_THREAT"]:
    if required_column not in gpr.columns:
        raise ValueError(f"Die GPR-Datei enthält die benötigte Spalte '{required_column}' nicht.")

gpr["Date"] = pd.to_datetime(gpr["Date"], errors="coerce")
gpr = gpr.dropna(subset=["Date"]).copy()
gpr = gpr.sort_values("Date").reset_index(drop=True)
gpr = gpr[["Date", "GPRD", "GPRD_ACT", "GPRD_THREAT"]].copy()

daily_join = stocks.merge(gpr, on="Date", how="left")
DAILY_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
daily_join.to_csv(DAILY_OUTPUT_PATH, index=False)

print(f"Loaded stocks shape: {stocks.shape}")
print(f"Loaded GPR shape: {gpr.shape}")
print(f"Daily join shape: {daily_join.shape}")
print(f"Used indices: {sorted(stocks['Index'].unique())}")

Loaded stocks shape: (104561, 8)
Loaded GPR shape: (15106, 4)
Daily join shape: (104561, 11)
Used indices: ['000001.SS', '399001.SZ', 'GDAXI', 'GSPTSE', 'HSI', 'IXIC', 'KS11', 'N100', 'N225', 'NYA', 'SSMI', 'TWII']


# 3. Feature Engineering

Hier wird der tägliche Datensatz zu einem monatlichen Panel verdichtet und um Lags, Leads, Volatilität, Z-Score, Krisendummy und Zielvariable erweitert.

In [3]:
stocks_monthly = (
    stocks.assign(YearMonth=stocks["Date"].dt.to_period("M"))
    .sort_values(["Index", "Date"])
    .groupby(["Index", "YearMonth"], as_index=False)
    .tail(1)
    .copy()
)
stocks_monthly["Close_month_end"] = stocks_monthly["Close"]
stocks_monthly = stocks_monthly.sort_values(["Index", "YearMonth"]).reset_index(drop=True)
stocks_monthly["stock_ret"] = stocks_monthly.groupby("Index")["Close_month_end"].pct_change() * 100
stocks_monthly["stock_ret_lag1"] = stocks_monthly.groupby("Index")["stock_ret"].shift(1)
stocks_monthly["stock_ret_lag2"] = stocks_monthly.groupby("Index")["stock_ret"].shift(2)
stocks_monthly["stock_ret_lag3"] = stocks_monthly.groupby("Index")["stock_ret"].shift(3)
stocks_monthly["Stock_vol12"] = (
    stocks_monthly.groupby("Index")["stock_ret"]
    .transform(lambda series: series.rolling(window=12, min_periods=12).std())
)

monthly_gpr = (
    gpr.assign(YearMonth=gpr["Date"].dt.to_period("M"))
    .groupby("YearMonth", as_index=False)[["GPRD", "GPRD_ACT", "GPRD_THREAT"]]
    .mean()
    .sort_values("YearMonth")
    .reset_index(drop=True)
)
monthly_gpr["gprd_ret"] = monthly_gpr["GPRD"].pct_change() * 100
monthly_gpr["gprd_act_ret"] = monthly_gpr["GPRD_ACT"].pct_change() * 100
monthly_gpr["gprd_threat_ret"] = monthly_gpr["GPRD_THREAT"].pct_change() * 100
monthly_gpr["GPR_zscore"] = (
    monthly_gpr["GPRD"] - monthly_gpr["GPRD"].rolling(window=24, min_periods=24).mean()
) / monthly_gpr["GPRD"].rolling(window=24, min_periods=24).std()

panel = stocks_monthly.merge(monthly_gpr, on="YearMonth", how="left")
panel = panel.sort_values(["Index", "YearMonth"]).reset_index(drop=True)
panel["gprd_ret_lead1"] = panel.groupby("Index")["gprd_ret"].shift(-1)
panel["gprd_ret_lead2"] = panel.groupby("Index")["gprd_ret"].shift(-2)
panel["gprd_ret_lead3"] = panel.groupby("Index")["gprd_ret"].shift(-3)

train_mask = monthly_gpr["YearMonth"] <= TRAIN_CUTOFF
train_gpr_returns = monthly_gpr.loc[train_mask, "gprd_ret"].dropna()
if train_gpr_returns.empty:
    raise ValueError("Für den Trainingszeitraum bis 2015-12 konnten keine GPR-Renditen für die Spike-Schwelle berechnet werden.")

GPR_SPIKE_THRESHOLD = train_gpr_returns.mean() + train_gpr_returns.std()
panel["GPR_spike"] = (panel["gprd_ret"] > GPR_SPIKE_THRESHOLD).astype(int)

region_labels = panel["Index"].map(REGION_MAP)
if region_labels.isna().any():
    missing_indices = sorted(panel.loc[region_labels.isna(), "Index"].unique())
    raise ValueError(f"Für folgende Indizes fehlt eine Regionszuordnung: {missing_indices}")

panel["Region"] = region_labels
region_encoder = LabelEncoder()
panel["Region_encoded"] = region_encoder.fit_transform(panel["Region"])
panel["Crisis_dummy"] = 0
for start_month, end_month in CRISIS_PERIODS:
    crisis_mask = (panel["YearMonth"] >= pd.Period(start_month, freq="M")) & (panel["YearMonth"] <= pd.Period(end_month, freq="M"))
    panel.loc[crisis_mask, "Crisis_dummy"] = 1

panel["target_gpr_up_lead1"] = (panel["gprd_ret_lead1"] > 0).astype(int)

print(f"Monthly panel shape before final filtering: {panel.shape}")
print(f"GPR_spike threshold: {GPR_SPIKE_THRESHOLD:.6f}")
print("GPR_spike distribution before final filtering:")
print(panel["GPR_spike"].value_counts(dropna=False).sort_index())
print("Monthly panel sample:")
display(panel[["YearMonth", "Index", "GPRD", "gprd_ret", "gprd_ret_lead1", "GPR_spike"]].head())

Monthly panel shape before final filtering: (5050, 30)
GPR_spike threshold: 43.966217
GPR_spike distribution before final filtering:
GPR_spike
0    4861
1     189
Name: count, dtype: int64
Monthly panel sample:


,YearMonth,Index,GPRD,gprd_ret,gprd_ret_lead1,GPR_spike
0,1997-07,000001.SS,47.708183,21.990898,1.528231,0
1,1997-08,000001.SS,48.437274,1.528231,-12.621124,0
2,1997-09,000001.SS,42.323946,-12.621124,16.849376,0
3,1997-10,000001.SS,49.455267,16.849376,21.454629,0
4,1997-11,000001.SS,60.065711,21.454629,-26.424023,0


# 4. Export

Der finale Datensatz wird bereinigt, auf den gewünschten Zeitraum begrenzt und als CSV gespeichert.

In [4]:
final_columns = [
    "YearMonth",
    "Index",
    "Region",
    "Region_encoded",
    "Close_month_end",
    "stock_ret",
    "stock_ret_lag1",
    "stock_ret_lag2",
    "stock_ret_lag3",
    "Stock_vol12",
    "GPRD",
    "GPRD_ACT",
    "GPRD_THREAT",
    "gprd_ret",
    "gprd_act_ret",
    "gprd_threat_ret",
    "gprd_ret_lead1",
    "gprd_ret_lead2",
    "gprd_ret_lead3",
    "GPR_zscore",
    "GPR_spike",
    "Crisis_dummy",
    "target_gpr_up_lead1",
]

final_dataset = panel.loc[:, final_columns].copy()
final_dataset = final_dataset[(final_dataset["YearMonth"] >= FINAL_START) & (final_dataset["YearMonth"] <= FINAL_END)].copy()
final_dataset = final_dataset.dropna(subset=[
    "Close_month_end",
    "stock_ret",
    "stock_ret_lag1",
    "stock_ret_lag2",
    "stock_ret_lag3",
    "Stock_vol12",
    "GPRD",
    "GPRD_ACT",
    "GPRD_THREAT",
    "gprd_ret",
    "gprd_act_ret",
    "gprd_threat_ret",
    "gprd_ret_lead1",
    "gprd_ret_lead2",
    "gprd_ret_lead3",
    "GPR_zscore",
    "GPR_spike",
    "Crisis_dummy",
    "target_gpr_up_lead1",
])

final_dataset["YearMonth"] = final_dataset["YearMonth"].astype(str)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
final_dataset.to_csv(FINAL_OUTPUT_PATH, index=False)

print(f"Final dataset saved to: {FINAL_OUTPUT_PATH}")
print(f"Final dataset shape: {final_dataset.shape}")

Final dataset saved to: d:\Anwendungsprojekt\data\processed\dataset_2001_2021.csv
Final dataset shape: (2908, 23)


# 5. Checks

Zum Schluss prüfen wir Form, Zeitraum, Indexabdeckung, fehlende Werte und die Zielverteilungen.

In [5]:
check_frame = final_dataset.copy()
check_frame["YearMonth_period"] = pd.PeriodIndex(check_frame["YearMonth"], freq="M")

print(f"Shape des finalen Datensatzes: {check_frame.shape}")
print(f"Startmonat: {check_frame['YearMonth_period'].min()}")
print(f"Endmonat: {check_frame['YearMonth_period'].max()}")
print(f"Anzahl der Indizes: {check_frame['Index'].nunique()}")
print(f"Verwendete Indizes: {sorted(check_frame['Index'].unique())}")
print("Fehlende Werte je Spalte:")
print(check_frame.drop(columns=["YearMonth_period"]).isna().sum())
print("Klassenverteilung von target_gpr_up_lead1:")
print(check_frame["target_gpr_up_lead1"].value_counts().sort_index())
print("Klassenverteilung von GPR_spike:")
print(check_frame["GPR_spike"].value_counts().sort_index())
print(f"GPR_spike threshold: {GPR_SPIKE_THRESHOLD:.6f}")
display(check_frame.head())

Shape des finalen Datensatzes: (2908, 24)
Startmonat: 2001-01
Endmonat: 2021-03
Anzahl der Indizes: 12
Verwendete Indizes: ['000001.SS', '399001.SZ', 'GDAXI', 'GSPTSE', 'HSI', 'IXIC', 'KS11', 'N100', 'N225', 'NYA', 'SSMI', 'TWII']
Fehlende Werte je Spalte:
YearMonth              0
Index                  0
Region                 0
Region_encoded         0
Close_month_end        0
stock_ret              0
stock_ret_lag1         0
stock_ret_lag2         0
stock_ret_lag3         0
Stock_vol12            0
GPRD                   0
GPRD_ACT               0
GPRD_THREAT            0
gprd_ret               0
gprd_act_ret           0
gprd_threat_ret        0
gprd_ret_lead1         0
gprd_ret_lead2         0
gprd_ret_lead3         0
GPR_zscore             0
GPR_spike              0
Crisis_dummy           0
target_gpr_up_lead1    0
dtype: int64
Klassenverteilung von target_gpr_up_lead1:
target_gpr_up_lead1
0    1500
1    1408
Name: count, dtype: int64
Klassenverteilung von GPR_spike:
GPR_spike
0  

,YearMonth,Index,Region,Region_encoded,Close_month_end,stock_ret,stock_ret_lag1,stock_ret_lag2,stock_ret_lag3,Stock_vol12,GPRD,GPRD_ACT,GPRD_THREAT,gprd_ret,gprd_act_ret,gprd_threat_ret,gprd_ret_lead1,gprd_ret_lead2,gprd_ret_lead3,GPR_zscore,GPR_spike,Crisis_dummy,target_gpr_up_lead1,YearMonth_period
42,2001-01,000001.SS,China_HK,1,2065.605957,-0.379608,0.138317,5.574200,2.676527,4.178811,47.345442,39.250704,53.581266,2.898991,-11.863157,14.962543,18.989928,9.558998,-20.567427,-1.127671,0,0,1,2001-01
43,2001-02,000001.SS,China_HK,1,1959.180054,-5.152285,-0.379608,0.138317,5.574200,3.632439,56.336307,45.958575,65.781575,18.989928,17.089810,22.769728,9.558998,-20.567427,22.282612,-0.552891,0,0,1,2001-02
44,2001-03,000001.SS,China_HK,1,2112.774902,7.839752,-5.152285,-0.379608,0.138317,3.980320,61.721494,57.449155,65.804611,9.558998,25.002039,0.035020,-20.567427,22.282612,18.331936,-0.164493,0,0,0,2001-03
45,2001-04,000001.SS,China_HK,1,2119.184082,0.303354,7.839752,-5.152285,-0.379608,3.987707,49.026970,47.124768,50.695351,-20.567427,-17.971348,-22.960792,22.282612,18.331936,-15.444990,-0.986328,0,0,1,2001-04
46,2001-05,000001.SS,China_HK,1,2214.257080,4.486302,0.303354,7.839752,-5.152285,4.061980,59.951460,56.564401,60.648585,22.282612,20.031149,19.633425,18.331936,-15.444990,6.919857,-0.080721,0,0,1,2001-05
